<a href="https://colab.research.google.com/github/Hashim123132/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10: Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hashim123132/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

Lane 1 (Ranking Signal Analysis) on the starter slice (30,000 rows, one row per content item). This notebook turns the validated Week-4 rule queue into a content action playbook: ranked actions, reason codes, archetype mapping, intended use, limits, human-review rules, cost and value, light monitoring triggers, and the exports the paper builds on next week. The learned model from Week 5 stays research-side: Week 6 showed it at chance on strictly pre-label-window features, so the operational queue is the rule.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the `writing-honest-claims` skill + `flyrank/flyrank-data` for this notebook.

## 1. Ranked actions + reason codes

**The queue.** Use the Week-4 rule to rank pages by visible search demand, position, and CTR gap. The queue is for human review, not automatic changes.

| Tier | Action                                     | Reason code                 |
| ---- | ------------------------------------------ | --------------------------- |
| P1   | Review title, meta, and SERP snippet first | `decline_risk_top_priority` |
| P2   | Consider a content refresh                 | `refresh_candidate`         |
| P3   | Monitor for another window                 | `monitor_only`              |
| None | No action                                  | `no_signal`                 |

**Reasoning:** Prioritize pages that have meaningful impressions, rank visibly, and capture fewer clicks than their position tier suggests.

**Archetype → action:**

* **CTR-starved:** Review title/meta and SERP intent.
* **Aging visible:** Consider refreshing facts, examples, and subtopics.
* **Young/small signal:** Monitor before changing.
* **Buried/no signal:** No action.

**Refresh insight:** Age alone is not a reliable refresh trigger in this snapshot. Use demand, position, and CTR gap as the main signals, with age as supporting context.


In [111]:
from pathlib import Path

ROOT = Path(".")
DATA = ROOT / "data" / "raw"
OUT = ROOT / "work" / "outputs"

DATA.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

print("Data folder:", DATA)
print("Output folder:", OUT)

Data folder: data/raw
Output folder: work/outputs


In [112]:
import pandas as pd, numpy as np, sklearn
from datetime import date
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

SEED = 42

print("sklearn", sklearn.__version__,
      "| numpy", np.__version__,
      "| pandas", pd.__version__)

df = pd.read_csv(DATA / "content_refresh_anonymized.csv")

df["down"] = (df.trend_direction == "down").astype(int)

v = df[(df.impressions_90d >= 500) & (df.avg_position > 0)].copy()

print("slice rows:", len(v), "| clients in slice:", v.client_id.nunique())

sklearn 1.6.1 | numpy 2.0.2 | pandas 2.2.2
slice rows: 16726 | clients in slice: 28


In [113]:
import pandas as pd, numpy as np, sklearn
from datetime import date
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

SEED = 42
print("sklearn", sklearn.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)

df = pd.read_csv(DATA / "content_refresh_anonymized.csv")
df["down"] = (df.trend_direction == "down").astype(int)   # eval only, never an input
v = df[(df.impressions_90d >= 500) & (df.avg_position > 0)].copy()
print("slice rows:", len(v), "| clients in slice:", v.client_id.nunique())

sklearn 1.6.1 | numpy 2.0.2 | pandas 2.2.2
slice rows: 16726 | clients in slice: 28


In [114]:
# The Week-4 rule, regenerated (same numbers as baseline_action_score.csv by design).
tier_ctr = v.groupby("position_tier").ctr.transform("median")
v["gap"] = (tier_ctr - v.ctr).clip(lower=0)
v["score"] = (v.avg_position <= 20).astype(int) * v.gap * np.log1p(v.impressions_90d)
flagged = v.score > 0
print("flagged (score > 0):", int(flagged.sum()), "of", len(v))

# Tier assignment: deterministic, observed columns only.
gap_top3 = v.loc[flagged, "score"].quantile(2 / 3)
p1 = flagged & (v.score >= gap_top3) & (v.impressions_90d >= 1000)
p2 = flagged & ~p1 & (v.content_age_days >= 180)
p3 = flagged & ~p1 & ~p2

v["priority_tier"] = np.select([p1, p2, p3], ["P1", "P2", "P3"], default="none")
v["reason_code"] = np.select(
    [p1, p2, p3],
    ["decline_risk_top_priority", "refresh_candidate", "monitor_only"],
    default="no_signal")
v["archetype"] = np.select(
    [p1, p2, p3],
    ["demand_held_ctr_starved", "aging_visible", "young_or_small_signal"],
    default="no_signal")
v["action"] = np.select([p1, p2, p3],
                        ["review_first", "refresh_plan", "monitor_only"],
                        default="no_action")

v = v.sort_values(["score", "impressions_90d"], ascending=[False, False])
print("tier mix (rows):")
print(v.groupby("priority_tier").size().to_string())
print("archetype mix (rows):")
print(v.groupby("archetype").size().to_string())

labels = v.down.values
def p_at_k(scores, k):
    order = np.argsort(-np.asarray(scores))
    return labels[order[:k]].mean()
print("\nmeasured precision@K (full slice, base rate %.3f):" % labels.mean())
print("  P@10 %.3f | P@20 %.3f | P@50 %.3f" % (p_at_k(v.score, 10), p_at_k(v.score, 20), p_at_k(v.score, 50)))

flagged (score > 0): 5885 of 16726
tier mix (rows):
priority_tier
P1       1649
P2       2525
P3       1711
none    10841
archetype mix (rows):
archetype
aging_visible               2525
demand_held_ctr_starved     1649
no_signal                  10841
young_or_small_signal       1711

measured precision@K (full slice, base rate 0.596):
  P@10 0.700 | P@20 0.700 | P@50 0.680


## 2. Intended use and limits

**Intended use.** An SEO or content reviewer uses this queue to decide which pages to review first. It is a prioritization tool, not a verdict or an automatic action trigger.

**Limits.**
- Results come from one snapshot and must be regenerated for a new export.
- The target is a proxy based on the observed last-30 vs previous-30 day trend.
- Week 6 found the future-safe ML model at chance (AUC 0.502), so no future-window ML ranking claim is made.
- The queue provides page-level signals; it does not explain query, competitor, or tracking effects.
- Human review is required before any action.

**Cost/value.** The queue focuses reviewer time on higher-priority pages. The cost is reviewer effort and possible false positives, so this is **decision-support, not automation**.

In [115]:
# The numbers the limits section rests on, recomputed here with the same splits and seeds as Weeks 5-6.
clients = v.client_id.drop_duplicates().to_numpy()
split_clients = np.random.default_rng(SEED).permutation(clients)
test_n = max(1, int(round(len(clients) * 0.2)))
test_cl = set(split_clients[:test_n])
is_test = v.client_id.isin(test_cl)
tr2, te2 = v[~is_test], v[is_test]

# Held-out rule numbers (same client holdout as Week 5).
tier_ctr_tr = tr2.groupby("position_tier").ctr.median()
rule_te = ((te2.avg_position <= 20).astype(int)
           * ((te2.position_tier.map(tier_ctr_tr)) - te2.ctr).clip(lower=0)
           * np.log1p(te2.impressions_90d))
def pk_at(s, k):
    order = np.argsort(-np.asarray(s))
    return te2.down.values[order[:k]].mean()
print("the rule, holdout test (n=%d, base %.3f):  P@10 %.3f | P@20 %.3f | P@50 %.3f" % (
    len(te2), te2.down.mean(), pk_at(rule_te, 10), pk_at(rule_te, 20), pk_at(rule_te, 50)))

# Week-5 logistic model, same holdout (research-side number only).
LOG_C = ["impressions_90d", "clicks_90d", "sessions_90d", "days_with_impressions",
         "word_count", "search_volume"]
RAW_C = ["avg_position", "ctr", "content_age_days", "days_since_last_update",
         "engagement_rate", "scroll_rate", "ai_traffic_pct"]
def features(d):
    x = pd.DataFrame(index=d.index)
    for c in LOG_C:
        x["log_" + c] = np.log1p(d[c])
    for c in RAW_C:
        x[c] = d[c]
    x["has_word_count"] = d.word_count.notna().astype(int)
    x["has_keyword_data"] = d.search_volume.notna().astype(int)
    for t in sorted(d.position_tier.unique()):
        x["tier_" + t] = (d.position_tier == t).astype(int)
    return x
Xtr, Xte = features(tr2), features(te2)
medf = Xtr.median(); Xtr, Xte = Xtr.fillna(medf), Xte.fillna(medf)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, random_state=SEED)).fit(
    Xtr, tr2.down.values)
p_lr = lr.predict_proba(Xte)[:, 1]
print("logistic model, same holdout:  P@10 %.3f | P@20 %.3f | P@50 %.3f | AUC %.3f (research only)" % (
    pk_at(p_lr, 10), pk_at(p_lr, 20), pk_at(p_lr, 50), roc_auc_score(te2.down.values, p_lr)))

# Week-6 audit number: clean time arm on the same data (AUC measured at chance).
for m in ["impressions", "clicks", "sessions"]:
    v["early_" + m] = (v[m + "_90d"] - v[m + "_last_30d"] - v[m + "_prev_30d"]).clip(lower=0)
v["down_past"] = ((v.impressions_prev_30d < v.early_impressions) & (v.impressions_prev_30d > 0)).astype(int)
okp = (v.impressions_prev_30d > 0).values
STATIC = ["word_count", "search_volume", "competition", "cpc", "content_age_days"]
def time_feats(d, window):
    cmap = {"early": {"clicks": "early_clicks", "sessions": "early_sessions"},
            "prev":  {"clicks": "clicks_prev_30d", "sessions": "sessions_prev_30d"}}[window]
    x = pd.DataFrame(index=d.index)
    for k, col in cmap.items():
        x["log_" + k] = np.log1p(d[col])
    for c in STATIC:
        x[c] = d[c]
    x["has_word_count"] = d.word_count.notna().astype(int)
    x["has_keyword_data"] = d.search_volume.notna().astype(int)
    for t in sorted(d.main_intent.dropna().unique()):
        x["intent_" + t] = (d.main_intent == t).astype(int)
    for t in sorted(d.competition_level.dropna().unique()):
        x["comp_" + t] = (d.competition_level == t).astype(int)
    return x
xtr_t, xte_t = time_feats(v, "early"), time_feats(v, "prev")
medt = xtr_t.median(); xtr_t, xte_t = xtr_t.fillna(medt), xte_t.fillna(medt)
lr_t = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, random_state=SEED)).fit(
    xtr_t[okp], v.down_past.values[okp])
p_t = lr_t.predict_proba(xte_t)[:, 1]
print("time-arm re-measured: AUC %.3f | base %.3f (Week-6 audit re-run here)" % (
    roc_auc_score(v.down.values, p_t), v.down.mean()))

the rule, holdout test (n=4219, base 0.617):  P@10 0.800 | P@20 0.900 | P@50 0.860
logistic model, same holdout:  P@10 0.700 | P@20 0.800 | P@50 0.760 | AUC 0.626 (research only)
time-arm re-measured: AUC 0.502 | base 0.596 (Week-6 audit re-run here)


## 3. Human review + the no-go list

**Before acting on any pick, a human must:**

1. Check the page title, meta, and search intent.
2. Review the current SERP and competitors for intent or ranking changes.
3. Verify impressions/clicks and rule out tracking issues.
4. Check whether the page was recently updated or is still ramping.
5. Decide whether to revise, refresh, or monitor.

**Never automate:**

* Do not automatically edit, delete, or deindex pages.
* Do not act on a single deep-ranking page (rank 21+).
* Do not treat P@K as a guarantee of future performance.
* Do not use label-derived features, IDs, or product flags as queue inputs.
* Re-run the playbook on each new data export.

**Final rule:** the notebook prioritizes pages; a human makes the final decision.


In [116]:
# The human-review rules and no-go list, printed as reference tables.
checklist = pd.DataFrame({
    "#": [1, 2, 3, 4, 5],
    "check": [
        "page text matches the query it ranks for",
        "the SERP intent fits the page (sub-intent mismatch explains many gaps)",
        "tracking consistency between last and previous windows (artifact check)",
        "recent edit or warm-up history (young pages ramp, not decay)",
        "only then: adjust text, plan a refresh, or watch one more window",
    ]})
print("HUMAN CHECKLIST")
print(checklist.to_string(index=False))

nogo = pd.DataFrame({"never": [
    "auto-edit titles, metas, or descriptions",
    "auto-delete or deindex any page",
    "act on a deep-rank (21+) page without extra evidence",
    "treat P values as future-window skill",
    "feed the queue trend ids, label rows, or any future-window data",
    "reuse this queue on a later export without regeneration",
]})
print("\nNO-GO LIST (never automate)")
print(nogo.to_string(index=False))

# The top of the queue, as the reviewer will see it (pseudonymized ids only).
cols = ["content_id", "priority_tier", "archetype", "reason_code", "position_tier",
        "avg_position", "ctr", "impressions_90d", "clicks_90d", "gap", "score", "action"]
print("\nP1 head of the queue (12 rows)")
print(v[v.priority_tier == "P1"][cols].head(12).to_string(index=False))

HUMAN CHECKLIST
 #                                                                   check
 1                                page text matches the query it ranks for
 2  the SERP intent fits the page (sub-intent mismatch explains many gaps)
 3 tracking consistency between last and previous windows (artifact check)
 4            recent edit or warm-up history (young pages ramp, not decay)
 5        only then: adjust text, plan a refresh, or watch one more window

NO-GO LIST (never automate)
                                                          never
                       auto-edit titles, metas, or descriptions
                                auto-delete or deindex any page
           act on a deep-rank (21+) page without extra evidence
                          treat P values as future-window skill
feed the queue trend ids, label rows, or any future-window data
        reuse this queue on a later export without regeneration

P1 head of the queue (12 rows)
          content_id prio

## 4. Monitoring / retrain triggers

The playbook has a declared half-life: it is valid between exports, and the triggers below say when it is stale.

| trigger | what to do | where the number comes from |
|---|---|---|
| new export arrives (any new snapshot refresh) | regenerate this notebook from scratch | export cadence, set by the pipeline |
| distribution shift in observed inputs | compare flagged share and tier mix of the new frame against the stored baseline; if the mix moves more than 5 points, review the rule before reuse | drift cells below |
| first measured future-window result | recompute P@K on the next real outcome window when it lands in the warehouse; a below-base result retires the rule | the audit methodology of Week 6 |
| reviewer feedback | reviewers record "confirmed declining" vs "healthy" per pick; if fewer than base + 5 pts of P1 picks get confirmed, pause the queue | from the human review log |
| model research (no dashboard use) | the Week-5 logistic model gets re-fit every export for exploration only and stays out of the export flavor | research notebooks |

The drift baseline lives in the exported metrics file, so the next run of this notebook can compare before writing anything.

In [117]:
OUT = "work/outputs"
os.makedirs(OUT, exist_ok=True)

queue_path = os.path.join(OUT, "action_queue.csv")
metrics_path = os.path.join(OUT, "playbook_metrics.json")

In [118]:
# Drift bookkeeping: store the baseline in the metrics json, compare on later runs.
import json, os
meta = os.path.join(OUT, "playbook_metrics.json")
baseline = {"flagged_share": float(flagged.mean()), "p1_rows": int(p1.sum()),
            "p2_rows": int(p2.sum()), "p3_rows": int(p3.sum()), "base_n": int(len(v))}
if os.path.exists(meta):
    prev = json.load(open(meta))
    b = prev.get("baseline", {})
    if b:
        delta = abs(b["flagged_share"] - baseline["flagged_share"])
        print("previous baseline found: flagged share %.3f" % b["flagged_share"])
        print("delta vs stored baseline: %.3f (>= 0.05 means the queue drifted; pause it)" % delta)
    else:
        print("no stored baseline yet; this run writes one")
else:
    print("no metrics json found\u2014this is the first run; baseline will be stored with the export")
print("baseline to store:", baseline)

previous baseline found: flagged share 0.352
delta vs stored baseline: 0.000 (>= 0.05 means the queue drifted; pause it)
baseline to store: {'flagged_share': 0.3518474231735023, 'p1_rows': 1649, 'p2_rows': 2525, 'p3_rows': 1711, 'base_n': 16726}


## 5. Exports for the paper

The exact files the paper builds on next week (paths, shapes, and checksums printed by the code):

- `work/outputs/action_queue.csv` - the playbook queue, sorted by rule score, with priority tier, archetype, reason code, and action. Regenerated in this notebook, so by design it stays out of git (the CI leak-guard blocks committed CSVs).
- `work/outputs/playbook_metrics.json` - the receipts every paper number traces back to: counts, tier and archetype mixes, measured precision@K (full slice and holdout), the audit numbers from Weeks 5-6, seeds, library versions, and the drift baseline.
- figures: the environment has no plotting library installed (offline), so chart-ready numbers live inside the JSON instead. The paper notebook can render them where a plotting library exists.

In [119]:
# Write the queue and the receipts; print the manifest with checksums.

import hashlib, os

os.makedirs(OUT, exist_ok=True)

queue_cols = ["content_id", "client_id", "position_tier", "avg_position", "ctr",
              "impressions_90d", "clicks_90d", "gap", "score", "priority_tier",
              "archetype", "reason_code", "action"]

queue_path = os.path.join(OUT, "action_queue.csv")

v[queue_cols].to_csv(queue_path, index=False)

with open(queue_path, "rb") as f:
    q_hash = hashlib.sha256(f.read()).hexdigest()

metrics = {
    "export": str(date.today()),
    "seed": SEED,
    "versions": {
        "sklearn": sklearn.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__
    },
    "slice": {
        "rows": int(len(v)),
        "clients": int(v.client_id.nunique()),
        "base_rate": round(float(v.down.mean()), 3),
        "flagged": int(flagged.sum())
    },
    "precision_at_k_full": {
        "k10": p_at_k(v.score, 10),
        "k20": p_at_k(v.score, 20),
        "k50": p_at_k(v.score, 50)
    },
    "rule_holdout": {
        "n": int(len(te2)),
        "base": float(te2.down.mean()),
        "k20": round(float(pk_at(rule_te, 20)), 3)
    },
    "model_research": {
        "k10": round(float(pk_at(p_lr, 10)), 3),
        "k20": round(float(pk_at(p_lr, 20)), 3),
        "AUC": round(float(roc_auc_score(te2.down.values, p_lr)), 3)
    },
    "audit_time": {
        "AUC": round(float(roc_auc_score(v.down.values, p_t)), 3),
        "base": round(float(v.down.mean()), 3)
    },
    "mix": {
        "tier": v.priority_tier.value_counts().to_dict(),
        "archetype": v.archetype.value_counts().to_dict()
    },
    "baseline": baseline,
    "action_queue_sha256": q_hash,
}

metrics_path = os.path.join(OUT, "playbook_metrics.json")

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("EXPORT MANIFEST")
print("  action_queue.csv      :", os.path.getsize(queue_path), "bytes | sha256", q_hash[:16])
print("  playbook_metrics.json :", os.path.getsize(metrics_path), "bytes")
print("note: queue csv stays out of git by design (CI leak-guard); the metrics JSON is the committed receipt.")

EXPORT MANIFEST
  action_queue.csv      : 2045888 bytes | sha256 1126bca86156f959
  playbook_metrics.json : 1037 bytes
note: queue csv stays out of git by design (CI leak-guard); the metrics JSON is the committed receipt.


## Self-check

- [x] Section 1 ranks actions with reason codes and the archetype mapping, decay/refresh insight included
- [x] Section 2 explains intended use, measured limits, and cost/value, careful wording throughout
- [x] Section 3 gives the human review checklist and the no-go list, what is never automated stated
- [x] Section 4 defines drift, future-label, reviewer, and cadence triggers
- [x] Section 5 exports the queue and the receipts, checksums printed
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere, only the badge link
- [x] Everything is decision-support, not production
- [x] Committed to my repo under `work/notebooks/`, then the repo URL goes on the card. Done.